In [34]:
import matplotlib.pyplot as plt
import numpy as np
import math
import os
import pandas as pd
from scipy import ndimage as ndi
from skimage.transform import rescale
from skimage.measure import marching_cubes, mesh_surface_area, label, regionprops, regionprops_table
from skimage import util, measure
import napari
from tifffile import imread
from liffile import LifFile
from pathlib import Path
import imagej
from imagej import Mode
import scyjava as sj
from matplotlib.colors import to_rgba

# import warnings
# warnings.filterwarnings("ignore")

ij = imagej.init("sc.fiji:fiji", mode=Mode.INTERACTIVE, add_legacy=True)

IJ = sj.jimport("ij.IJ")
Duplicator = sj.jimport("ij.plugin.Duplicator")
WekaSegmentation = sj.jimport("trainableSegmentation.WekaSegmentation")
ImagePlus = sj.jimport("ij.ImagePlus")
ImageStack = sj.jimport("ij.ImageStack")
FloatProcessor = sj.jimport("ij.process.FloatProcessor")

In [35]:
def numpy_to_imageplus(arr: np.ndarray, title="image") -> ImagePlus:
    if arr.ndim == 2:
        y, x = arr.shape
        pix = np.asarray(arr, dtype=np.float32).ravel()
        java_floats = sj.jarray("f", pix.size)
        # fill java float[]
        for i, v in enumerate(pix.tolist()):
            java_floats[i] = float(v)
        fp = FloatProcessor(x, y, java_floats)
        return ImagePlus(title, fp)

    if arr.ndim == 3:
        z, y, x = arr.shape
        stack = ImageStack(x, y)
        for zi in range(z):
            pix = np.asarray(arr[zi], dtype=np.float32).ravel()
            java_floats = sj.jarray("f", pix.size)
            for i, v in enumerate(pix.tolist()):
                java_floats[i] = float(v)
            fp = FloatProcessor(x, y, java_floats)
            stack.addSlice(fp)
        return ImagePlus(title, stack)

    raise ValueError(f"Unsupported shape: {arr.shape}")

In [36]:
def imageplus_to_numpy(imp: ImagePlus) -> np.ndarray:
    w, h = imp.getWidth(), imp.getHeight()
    n = imp.getStackSize()
    stack = imp.getStack()
    out = []
    for z in range(1, n + 1):
        ip = stack.getProcessor(z)
        pix = np.array(ip.getPixels()).reshape(h, w)
        out.append(pix)
    return np.stack(out, axis=0) if n > 1 else out[0]

In [37]:
def convert(img, target_type_min, target_type_max, target_type):
    """
    Converts an image to a specified data type while scaling its intensity values.

    This function rescales the intensity values of an image from its original range 
    to a new target range specified by `target_type_min` and `target_type_max`, and 
    then converts it to the desired data type.

    This step is required as deconvolved images are not always scaled 0->255! 

    Parameters:
    -----------
    img : numpy.ndarray
        The input image array to be converted.
    target_type_min : int or float
        The minimum value of the target intensity range.
    target_type_max : int or float
        The maximum value of the target intensity range.
    target_type : numpy.dtype
        The desired data type of the output image (e.g., np.uint8, np.float32).

    Returns:
    --------
    new_img : numpy.ndarray
        The rescaled image with values mapped to the new intensity range and converted 
        to the specified data type.

    Notes:
    ------
    - This function performs a linear transformation to scale pixel values.
    - It ensures that the output values are properly mapped between `target_type_min` and 
      `target_type_max`.
    """
    imin = img.min()
    imax = img.max()

    a = (target_type_max - target_type_min) / (imax - imin)
    b = target_type_max - a * imax
    new_img = (a * img + b).astype(target_type)
    return new_img

In [38]:
def apply_weka_with_exact_preprocessing(image: np.ndarray, model_path: str) -> np.ndarray:
    imp = numpy_to_imageplus(image, title="image")
    dup = Duplicator().run(imp)
    IJ.run(dup, "8-bit", "")
    IJ.run(dup, "Auto Threshold", "method=Otsu stack")
    IJ.run(dup, "Erode (3D)", "iso=255")

    try:
        seg = WekaSegmentation(dup)
        seg.loadClassifier(model_path)
        out_imp = seg.applyClassifier(dup, 0, False)
        if out_imp is None:
            out_imp = seg.getClassifiedImage()

        if out_imp is None:
            raise RuntimeError("No classified image returned (out_imp is None). Likely a Java-side error or model/input mismatch.")

        return imageplus_to_numpy(out_imp)

    except Exception:
        # Show Java exception details if present
        print("=== Python/Java exception ===")
        raise
    finally:
        try:
            IJ.run("Close All")
        except Exception:
            pass

In [39]:
def step(axis, xa):
    if axis not in xa.coords or xa.coords[axis].size < 2:
        return None
    return float(xa.coords[axis][1] - xa.coords[axis][0])

In [40]:
def segment_and_quantify(i, lif, lif_path, classifier_path):
    img = lif.images[i]
    lif_name = "".join(os.path.basename(lif_path).lower().replace(".lif", ""))
    image_name = "".join(img.path)
    # print(lif_name, image_name)
    try:
        if img.dims == ('T', 'Z', 'Y', 'X'):
            image = img.asarray()
            xa = img.asxarray()

            # liffile coordinates are typically in meters → convert to µm
            x_um = step("X", xa) * 1e6 if step("X", xa) is not None else None
            # y_um = step("Y", xa) * 1e6 if step("Y", xa) is not None else None
            z_um = step("Z", xa) * 1e6 if step("Z", xa) is not None else None
            # spacing_um = [x_um, y_um, z_um]

            t0 = image[0,:,:,:]
            t2 = image[2,:,:,:]
            gel_matrix = apply_weka_with_exact_preprocessing(t0, classifier_path)
        
            vasculature_segmentation = (gel_matrix == 0).astype(int)
            vasculature_labels = label(vasculature_segmentation)

            table = regionprops_table(vasculature_labels, properties=('label', 'area'),)

            condition = (table['area'] >= 20)
            input_labels = table['label']
            output_labels = input_labels * condition
            output_labels = util.map_array(vasculature_labels, input_labels, output_labels)
            clean_vasculature_segmentation = output_labels > 0
            # clean_gel_segmentation = (clean_vasculature_segmentation == 0).astype(int)
            
            t2 = convert(t2, 0, 255, np.uint8)
            t0 = convert(t0, 0, 255, np.uint8)

            final_vascular_intensity = np.float64(np.sum(t2[clean_vasculature_segmentation==1]))
            final_gel_intensity = np.float64(np.sum(t2[clean_vasculature_segmentation==0]))
            initial_vascular_intensity = np.float64(np.sum(t0[clean_vasculature_segmentation==1]))
            initial_gel_intensity = np.float64(np.sum(t0[clean_vasculature_segmentation==0]))

            rescaled_t0 = rescale(scale = (z_um/x_um,1,1), image=(t0 ), anti_aliasing = False)
            # rescaled_t2 = rescale(scale = (z_um/x_um,1,1), image=(t2 ), anti_aliasing = False)
            rescaled_vasculature_segmentation = rescale(scale = (z_um/x_um,1,1), image=(clean_vasculature_segmentation ), anti_aliasing = False, order=0, preserve_range=True).astype(clean_vasculature_segmentation.dtype)
            verts, faces, _, _ = marching_cubes(rescaled_vasculature_segmentation.astype(np.uint8), level=0.5, spacing=(x_um,) * 3)
            vasculature_surface_area = mesh_surface_area(verts, faces)
            vascular_volume = np.sum(rescaled_vasculature_segmentation==1) * ((x_um)**3)
            total_volume = rescaled_t0.shape[0] * rescaled_t0.shape[1] * rescaled_t0.shape[2] * ((x_um)**3)
            bleaching_coefficient = initial_vascular_intensity/final_vascular_intensity
            gel_volume = total_volume - vascular_volume
            p=(1/360)*(gel_volume/vasculature_surface_area)*(((bleaching_coefficient*final_gel_intensity)-initial_gel_intensity)/((initial_vascular_intensity)-(initial_gel_intensity)))
            output = pd.DataFrame({
                "lif_name": [lif_name],
                "image_name": [image_name],
                "image_shape": [image.shape],
                "final_gel_intensity": [final_gel_intensity],
                "final_vascular_intensity": [final_vascular_intensity],
                "initial_gel_intensity": [initial_gel_intensity],
                "initial_vascular_intensity": [initial_vascular_intensity],
                "vascular_volume_um3": [vascular_volume],
                "gel_volume_um3": [gel_volume],
                "vasculature_surface_area_um2": [vasculature_surface_area],
                "bleaching_coefficient": [bleaching_coefficient],
                "p_um/s": [p],
                "p_cm/s": [p*0.0001],
                })
        else:
            output = pd.DataFrame({
                "lif_name": [lif_name],
                "image_name": [image_name],
                "flag": ["IMAGE FAILED"],})
    except Exception:
        output = pd.DataFrame({
            "lif_name": [lif_name],
            "image_name": [image_name],
            "flag": ["CZYX Image"],})
    finally:
        try:
            IJ.run("Close All")
        except Exception:
            pass
    return output


In [ ]:
from typing import Dict, List, Optional
from pathlib import Path

import pandas as pd
from liffile import LifFile
from magicgui import magicgui
from magicgui.widgets import Container, TextEdit
from IPython.display import display


class PerfusionNapariGUIApp:
    def __init__(self):
        self.viewer = napari.Viewer()

        self._selected_lif_path: Optional[Path] = None
        self._classifier_path: Optional[Path] = None
        self._image_choice_map: Dict[str, int] = {}
        self._results: List[pd.DataFrame] = []
        self._results_df: Optional[pd.DataFrame] = None

        self.log_output = TextEdit(value="")
        self.log_output.min_height = 140
        self.log_output.max_height = 320
        try:
            self.log_output.native.setReadOnly(True)
        except Exception:
            pass

        self.load_images = magicgui(
            self._list_images,
            lif_path={"label": "Select .lif", "mode": "r", "filter": "*.lif"},
            classifier_path={"label": "Weka classifier", "mode": "r"},
            call_button="Load images",
        )

        self.run_single = magicgui(
            self._run_single_image,
            image_choice={"label": "Image", "choices": ["(load images)"], "widget_type": "ComboBox"},
            call_button="Analyse Single Image",
        )

        self.run_all = magicgui(
            self._run_all_images,
            output_csv={"label": "Output CSV", "mode": "w", "value": "perfusion_all_images.csv"},
            call_button="Analyse All Images",
        )

        self.save_results = magicgui(
            self._save_results,
            output_csv={"label": "Save current results", "mode": "w", "value": "perfusion_results.csv"},
            call_button="Save This Result",
        )

        log_panel = Container(widgets=[self.log_output])
        self.viewer.window.add_dock_widget(self.load_images, area="right")
        self.viewer.window.add_dock_widget(self.run_single, area="right")
        self.viewer.window.add_dock_widget(self.run_all, area="right")
        self.viewer.window.add_dock_widget(self.save_results, area="right")
        self.viewer.window.add_dock_widget(log_panel, area="right")

    def _append_log(self, message: str):
        self.log_output.value = (
            self.log_output.value.rstrip() + "\n" + message if self.log_output.value else message
        )

    @staticmethod
    def _normalize_csv_path(path_value: Path) -> Path:
        path = Path(path_value)
        if path.suffix.lower() != ".csv":
            return path.with_suffix(".csv")
        return path

    def _clear_preview_layers(self):
        for layer_name in ["t0", "t2", "Segmented Vasculature"]:
            if layer_name in self.viewer.layers:
                self.viewer.layers.remove(self.viewer.layers[layer_name])

    def _build_preview_from_lif_image(self, lif_img, classifier_path: str):
        if lif_img.dims != ("T", "Z", "Y", "X"):
            raise ValueError("Preview currently supports TZYX images only")

        image = lif_img.asarray()
        t0 = image[0, :, :, :]
        t2 = image[2, :, :, :]

        gel_matrix = apply_weka_with_exact_preprocessing(t0, classifier_path)
        vasculature_segmentation = (gel_matrix == 0).astype(int)
        vasculature_labels = label(vasculature_segmentation)
        table = regionprops_table(vasculature_labels, properties=("label", "area"))

        condition = table["area"] >= 20
        input_labels = table["label"]
        output_labels = input_labels * condition
        output_labels = util.map_array(vasculature_labels, input_labels, output_labels)
        clean_vasculature_segmentation = output_labels > 0

        return t0, t2, clean_vasculature_segmentation

    def _show_single_result_layers(self, t0, t2, clean_vasculature_segmentation):
        self._clear_preview_layers()
        self.viewer.add_image(t0, name="t0")
        self.viewer.add_image(t2, name="t2")
        self.viewer.add_labels(
            clean_vasculature_segmentation,
            name="Segmented Vasculature",
            colormap={1: np.array(to_rgba("red"), dtype=float)},
        )

    def _require_loaded_inputs(self) -> bool:
        if self._selected_lif_path is None or not self._selected_lif_path.exists():
            self._append_log("[WARN] Select a .lif and click Load images first.")
            return False
        if self._classifier_path is None or not self._classifier_path.exists():
            self._append_log("[WARN] Select a valid Weka classifier and click Load images.")
            return False
        return True

    def _list_images(self, lif_path: Path = Path(), classifier_path: Path = Path()):
        if not lif_path or not Path(lif_path).exists():
            self._append_log("[WARN] Select a valid .lif file.")
            return None
        if not classifier_path or not Path(classifier_path).exists():
            self._append_log("[WARN] Select a valid Weka classifier.")
            return None

        self._selected_lif_path = Path(lif_path)
        self._classifier_path = Path(classifier_path)
        labels: List[str] = []
        choice_map: Dict[str, int] = {}

        try:
            with LifFile(self._selected_lif_path) as lif:
                for i, img in enumerate(lif.images):
                    name = "".join(getattr(img, "path", ())) or f"Image {i}"
                    label = f"{i}: {name}"
                    labels.append(label)
                    choice_map[label] = i
        except Exception as e:
            self._append_log(f"[ERROR] Failed reading lif: {type(e).__name__}: {e}")
            return None

        if not labels:
            self._image_choice_map = {}
            self.run_single.image_choice.choices = ["(no images)"]
            self.run_single.image_choice.value = "(no images)"
            self._append_log("[WARN] No images found in this .lif.")
            return None

        self._image_choice_map = choice_map
        self.run_single.image_choice.choices = labels
        self.run_single.image_choice.value = labels[0]
        self._append_log(f"[OK] Loaded .lif with {len(labels)} image(s).")
        self._append_log(f"[OK] Using classifier: {self._classifier_path.name}")
        return labels

    def _run_single_image(self, image_choice: str = "(load images)"):
        if not self._require_loaded_inputs():
            return None
        if image_choice not in self._image_choice_map:
            self._append_log("[WARN] Select an image from dropdown.")
            return None

        image_index = self._image_choice_map[image_choice]
        self._append_log(f"[PROGRESS] Running single image_index={image_index}...")

        try:
            with LifFile(self._selected_lif_path) as lif:
                row = segment_and_quantify(image_index, lif, self._selected_lif_path, str(self._classifier_path))

                try:
                    lif_img = lif.images[image_index]
                    t0, t2, clean_vasculature_segmentation = self._build_preview_from_lif_image(
                        lif_img, str(self._classifier_path)
                    )
                    self._show_single_result_layers(
                        t0,
                        t2,
                        clean_vasculature_segmentation,
                    )
                    self._append_log("[OK] Added t0/t2 and Segmented Vasculature layer to napari viewer.")
                except Exception as e:
                    self._append_log(
                        f"[WARN] Could not render preview layers: {type(e).__name__}: {e}"
                    )

        except Exception as e:
            self._append_log(f"[ERROR] Single image failed: {type(e).__name__}: {e}")
            return None

        self._results.append(row)
        self._results_df = pd.concat(self._results, ignore_index=True)
        self._append_log("[OK] Single image complete.")
        display(self._results_df)
        return row

    def _run_all_images(self, output_csv: Path = Path("perfusion_all_images.csv")):
        if not self._require_loaded_inputs():
            return None

        output_csv = self._normalize_csv_path(output_csv)
        output_csv.parent.mkdir(parents=True, exist_ok=True)

        rows: List[pd.DataFrame] = []
        try:
            with LifFile(self._selected_lif_path) as lif:
                n_images = len(lif.images)
                self._append_log(
                    f"[INFO] Running all images in {self._selected_lif_path.name} (n={n_images})"
                )
                for i in range(n_images):
                    self._append_log(f"[PROGRESS] ({i + 1}/{n_images}) image_index={i}")
                    try:
                        row = segment_and_quantify(i, lif, self._selected_lif_path, str(self._classifier_path))
                    except Exception as e:
                        row = pd.DataFrame(
                            {
                                "lif_name": [self._selected_lif_path.stem],
                                "image_name": [f"image_{i}"],
                                "flag": [f"FAILED: {type(e).__name__}: {e}"],
                            }
                        )
                    rows.append(row)
        except Exception as e:
            self._append_log(f"[ERROR] Batch failed: {type(e).__name__}: {e}")
            return None

        if not rows:
            self._append_log("[WARN] No results generated.")
            return None

        batch_df = pd.concat(rows, ignore_index=True)
        batch_df.to_csv(output_csv, index=False)

        self._results.append(batch_df)
        self._results_df = pd.concat(self._results, ignore_index=True)

        self._append_log(f"[OK] Saved all-image CSV to {output_csv}")
        display(batch_df)
        return batch_df

    def _save_results(self, output_csv: Path = Path("perfusion_results.csv")):
        if self._results_df is None or self._results_df.empty:
            self._append_log("[WARN] No results to save yet.")
            return None

        output_csv = self._normalize_csv_path(output_csv)
        output_csv.parent.mkdir(parents=True, exist_ok=True)
        self._results_df.to_csv(output_csv, index=False)
        self._append_log(f"[OK] Saved current results to {output_csv}")
        return output_csv


app = PerfusionNapariGUIApp()

c:\Users\taylorhearn\AppData\Local\miniconda3\envs\nap-ij\Lib\site-packages\napari\utils\colormaps\colormap.py:435: UserWarning: color_dict did not provide a default color. Missing keys will be transparent. To provide a default color, use the key `None`, or provide a defaultdict instance.
  warn(


,lif_name,image_name,image_shape,final_gel_intensity,final_vascular_intensity,initial_gel_intensity,initial_vascular_intensity,vascular_volume_um3,gel_volume_um3,vasculature_surface_area_um2,bleaching_coefficient,p_um/s,p_cm/s
0,m4_2024.10.24_fl2_fl6,rLN2_24.10.24_device1/P 2,"(3, 36, 512, 512)",15246285.0,116276721.0,11906772.0,124558040.0,7.342314e+07,1.704046e+08,5.396960e+06,1.071221,0.003445,3.445425e-07
